# agent_framework_orchestrations

## Example 1: a sequential pipeline runs each agent in turn

In [ ]:
# Example 1: SequentialBuilder chains a writer and a reviewer; the workflow's output is the final conversation
from agent_framework import Agent, ChatResponse, ChatResponseUpdate, Content, Message, ResponseStream
from agent_framework_orchestrations import SequentialBuilder

class CannedChatClient:
    """Stands in for a model: the same answer, streamed or not (the agent may ask for either)."""
    additional_properties: dict = {}
    def __init__(self, text):
        self.text = text
    def get_response(self, messages, *, stream=False, options=None, **kwargs):
        if stream:
            async def updates():
                yield ChatResponseUpdate(contents=[Content.from_text(self.text)], role="assistant")
            return ResponseStream(updates(), finalizer=ChatResponse.from_updates)
        async def answer():
            return ChatResponse(messages=Message(role="assistant", contents=[self.text]))
        return answer()

writer = Agent(client=CannedChatClient("a draft haiku"), name="writer")
reviewer = Agent(client=CannedChatClient("approved"), name="reviewer")
result = await SequentialBuilder(participants=[writer, reviewer]).build().run("write a haiku")
outputs = result.get_outputs()
assert len(outputs) == 1 and "approved" in outputs[0].text, outputs
print("Example 1:", type(result).__name__, "->", outputs[0].text)

## Example 2: a concurrent fan-out asks every agent at once

In [ ]:
# Example 2: ConcurrentBuilder sends the same task to two agents and aggregates their answers
from agent_framework import Agent
from agent_framework_orchestrations import ConcurrentBuilder

optimist = Agent(client=CannedChatClient("it will work"), name="optimist")
pessimist = Agent(client=CannedChatClient("it will fail"), name="pessimist")
result = await ConcurrentBuilder(participants=[optimist, pessimist]).build().run("will it work?")
text = " ".join(output.text for output in result.get_outputs())
assert "it will work" in text and "it will fail" in text, text
print("Example 2:", text)

## Example 3: the builders refuse an empty participant list

In [ ]:
# Example 3: building a pipeline with no agents is a ValueError, not a silent no-op
from agent_framework_orchestrations import ConcurrentBuilder, SequentialBuilder

for builder in (SequentialBuilder, ConcurrentBuilder):
    try:
        builder(participants=[]).build()
    except ValueError as error:
        print("Example 3:", builder.__name__, "refused:", error)
    else:
        raise AssertionError(f"{builder.__name__} built with no participants")

## Example 4: a group chat with a round-robin selection function

In [ ]:
# Example 4: GroupChatBuilder picks the next speaker with our function and stops after max_rounds
from agent_framework import Agent
from agent_framework_orchestrations import GroupChatBuilder

alice = Agent(client=CannedChatClient("alice speaks"), name="alice")
bob = Agent(client=CannedChatClient("bob speaks"), name="bob")
order = []
def round_robin(state):
    name = ["alice", "bob"][len(order) % 2]
    order.append(name)
    return name
workflow = GroupChatBuilder(participants=[alice, bob], selection_func=round_robin, max_rounds=2).build()
result = await workflow.run("discuss")
assert order[:2] == ["alice", "bob"], order
print("Example 4: speakers", order, "outputs", len(result.get_outputs()))

## Example 5: the umbrella namespace re-exports the same builders

In [ ]:
# Example 5: agent_framework.orchestrations is a lazy alias of this package (core's re-export of the connector)
import importlib.metadata
import agent_framework.orchestrations as via_core
import agent_framework_orchestrations as direct

assert via_core.SequentialBuilder is direct.SequentialBuilder
assert via_core.MagenticBuilder is direct.MagenticBuilder
print("Example 5:", importlib.metadata.version("agent-framework-orchestrations"), "re-exported through agent_framework.orchestrations")